In [1]:
# STEP 1 Upload the CSV file

from google.colab import files
uploaded = files.upload()

Saving tickets_raw.csv to tickets_raw.csv


In [2]:
# STEP 2: Import libraries and load dataset

import pandas as pd
import numpy as np
import re

df = pd.read_csv('tickets_raw.csv')

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully!
Shape: (1800, 14)


,ticket_id,created_at,channel,requester_dept,location,priority,short_description,description,category,assignment_group,status,resolved_at,resolution_hours,reopened
0,INC100000,2026-04-01 05:26:00,Phone,Legal,Bengaluru,P4,DNS issue,"Hi, The intranet portal doesn't load. It start...",Network,Network Operations,Resolved,2026-04-02 23:08:00,41.7,False
1,INC100001,2026-04-01 09:43:00,Portal,HR,Hyderabad,P3,account creation,"dear support, please set up accounts for two c...",Access,Identity & Access Management,Resolved,2026-04-01 15:01:00,5.3,False
2,INC100002,2026-04-01 10:04:00,Portal,Sales,Bengaluru,P3,Excel closes itself,"Good morning, Excel stops responding when I ex...",Software,Application Support,Resolved,2026-04-01 22:28:00,12.4,False
3,INC100003,2026-04-01 12:18:00,Email,Operations,Chennai,P4,quota exceeded,I get a mailbox almost full warning. Clients' ...,Email & Collaboration,Messaging & Collaboration,Resolved,2026-04-01 20:30:00,8.2,False
4,INC100004,2026-04-01 15:12:00,Email,Marketing,Pune,P3,new hire setup,"Hi IT, A new team member joins on Monday and n...",Access,Identity & Access Management,Resolved,2026-04-01 19:42:00,4.5,False


In [3]:
# STEP 3: Missing value analysis

missing_count = df.isnull().sum()

missing_percentage = (
    df.isnull().sum() / len(df) * 100
).round(2)

missing_summary = pd.DataFrame({
    'Missing Values': missing_count,
    'Missing Percentage': missing_percentage
})

display(missing_summary)

,Missing Values,Missing Percentage
ticket_id,0,0.00
created_at,0,0.00
channel,30,1.67
requester_dept,38,2.11
location,47,2.61
priority,35,1.94
short_description,0,0.00
description,43,2.39
category,0,0.00
assignment_group,0,0.00


In [4]:
# STEP 4: Duplicate analysis

# Check completely duplicated rows
print("Fully duplicated rows:", df.duplicated().sum())

# Check duplicate ticket IDs
print("Duplicate ticket IDs:", df['ticket_id'].duplicated().sum())

# Check repeated short descriptions
print(
    "Repeated short descriptions:",
    df['short_description'].duplicated().sum()
)

# Check repeated detailed descriptions
print(
    "Repeated descriptions:",
    df['description'].duplicated().sum()
)

Fully duplicated rows: 0
Duplicate ticket IDs: 0
Repeated short descriptions: 1264
Repeated descriptions: 61


In [16]:
#Handle missing values only in model-required

df['short_description'] = df['short_description'].fillna('')
df['description'] = df['description'].fillna('')

# Handle missing priority values
df['priority'] = df['priority'].fillna('Unknown')

# Check target column
print("Missing values in category:", df['category'].isnull().sum())

# Verify missing values
print("\nMissing values after handling:")
print(df[['short_description', 'description', 'priority', 'category']].isnull().sum())

Missing values in category: 0

Missing values after handling:
short_description    0
description          0
priority             0
category             0
dtype: int64


In [12]:
#Combine the text columns
df['clean_text'] = (
    df['short_description'] + ' ' +
    df['description']
).str.strip()

df[['clean_text', 'category']].head()

,model_text,category
0,"DNS issue Hi, The intranet portal doesn't load...",Network
1,"account creation dear support, please set up a...",Access
2,"Excel closes itself Good morning, Excel stops ...",Software
3,quota exceeded I get a mailbox almost full war...,Email & Collaboration
4,"new hire setup Hi IT, A new team member joins ...",Access


In [15]:
#Clean the combined text
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['clean_text'].apply(clean_text)

In [17]:
# Handle missing priority values

df['priority'] = df['priority'].fillna('Unknown')

# Verify
print(df['priority'].isnull().sum())
print(df['priority'].value_counts())

0
priority
P3         858
P2         445
P4         346
P1         116
Unknown     35
Name: count, dtype: int64


In [19]:
# Select columns for our project

selected_columns = [
    'ticket_id',
    'created_at',
    'priority',
    'requester_dept',
    'model_text',
    'category',
    'assignment_group',
    'status',
    'resolution_hours'
]

df_cleaned = df[selected_columns].copy()

df_cleaned.head()

,ticket_id,created_at,priority,requester_dept,model_text,category,assignment_group,status,resolution_hours
0,INC100000,2026-04-01 05:26:00,P4,Legal,"dns issue hi, the intranet portal doesn't load...",Network,Network Operations,Resolved,41.7
1,INC100001,2026-04-01 09:43:00,P3,HR,"account creation dear support, please set up a...",Access,Identity & Access Management,Resolved,5.3
2,INC100002,2026-04-01 10:04:00,P3,Sales,"excel closes itself good morning, excel stops ...",Software,Application Support,Resolved,12.4
3,INC100003,2026-04-01 12:18:00,P4,Operations,quota exceeded i get a mailbox almost full war...,Email & Collaboration,Messaging & Collaboration,Resolved,8.2
4,INC100004,2026-04-01 15:12:00,P3,Marketing,"new hire setup hi it, a new team member joins ...",Access,Identity & Access Management,Resolved,4.5


In [20]:
# Save cleaned dataset

df_cleaned.to_csv(
    'IT_Support_Tickets_Cleaned.csv',
    index=False
)

# Download CSV
from google.colab import files
files.download('IT_Support_Tickets_Cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>